# Phonon Mode Pictures (Arrows, No Animation)
Static arrow plots derived from the `phonon_gif.ipynb` workflow.

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

# ============================================================
# Inputs
# ============================================================
SMODES_FILE = Path("../../data/MnBi2Te4/phonon") / 'SMODES.out'
IRREP_ANCHOR = 'Irrep T2-'
OUT_DIR = Path("../../data/MnBi2Te4/phonon") / 'pics'
OUT_DIR.mkdir(exist_ok=True)

# arrow/plot styling
PEAK_DISPLACEMENT = 3
MIN_VISIBLE_NONZERO = 0.30
ARROW_COLOR = "#02DEB9"
FIGSIZE = (3.4, 7.4)
DPI = 180
STATIONARY_ALPHA = 0.15
# rough visual radii for marker sizing; chosen for readability, not physical accuracy
ATOM_DISPLAY_RADII = {'Mn': 1.20, 'Bi': 1.55, 'Te': 1.40}
ATOM_SIZE_SCALE = 40

ATOM_LINE_RE = re.compile(r'^\s*(\d+)\s+([A-Za-z0-9_]+)\s+([-+0-9.]+)\s+([-+0-9.]+)\s+([-+0-9.]+)\s*$')

def extract_irrep_section(smodes_text: str, irrep_anchor: str) -> str:
    lines = [ln.rstrip('\n') for ln in smodes_text.splitlines()]
    start = None
    for i, ln in enumerate(lines):
        if irrep_anchor in ln:
            start = i
            break
    if start is None:
        raise ValueError(f"Could not find irrep anchor: {irrep_anchor}")

    atom_header = None
    for i in range(start, len(lines)):
        if lines[i].strip().startswith('atom  type   position'):
            atom_header = i
            break
    if atom_header is None:
        raise ValueError(f"Could not find atom header after {irrep_anchor}")

    end = len(lines)
    for i in range(atom_header + 1, len(lines)):
        if lines[i].startswith('***********************************************'):
            end = i
            break

    return '\n'.join(lines[atom_header:end])

def parse_atoms_and_modes(smodes_text: str):
    lines = [ln.rstrip('\n') for ln in smodes_text.splitlines()]

    atom_ids, atom_types, atom_positions = [], [], []
    i = 0
    while i < len(lines) and 'Symmetry modes:' not in lines[i]:
        m = ATOM_LINE_RE.match(lines[i])
        if m:
            atom_ids.append(int(m.group(1)))
            atom_types.append(m.group(2))
            atom_positions.append([float(m.group(3)), float(m.group(4)), float(m.group(5))])
        i += 1

    atom_ids = np.array(atom_ids, dtype=int)
    atom_types = np.array(atom_types, dtype=object)
    atom_positions = np.array(atom_positions, dtype=float)

    while i < len(lines) and 'Symmetry modes:' not in lines[i]:
        i += 1
    if i == len(lines):
        raise ValueError("Could not find 'Symmetry modes:' in SMODES text.")
    i += 1

    mode_blocks = []
    current = []

    def flush():
        nonlocal current
        if current:
            mode_blocks.append(current)
            current = []

    while i < len(lines):
        ln = lines[i].strip()
        if ln.startswith('-'):
            flush()
            i += 1
            continue
        m = ATOM_LINE_RE.match(lines[i])
        if m:
            atom_id = int(m.group(1))
            atype = m.group(2)
            disp = np.array([float(m.group(3)), float(m.group(4)), float(m.group(5))], dtype=float)
            current.append((atom_id, atype, disp))
        i += 1

    flush()
    return atom_ids, atom_types, atom_positions, mode_blocks

def build_symmetry_mode_displacements(atom_ids, mode_blocks):
    n = len(atom_ids)
    id_to_idx = {aid: idx for idx, aid in enumerate(atom_ids)}
    dz_sym = []
    for block in mode_blocks:
        dz = np.zeros(n, dtype=float)
        for atom_id, _, disp_xyz in block:
            dz[id_to_idx[atom_id]] = disp_xyz[2]
        dz_sym.append(dz)
    return np.array(dz_sym, dtype=float)

def _style_key(t):
    t = str(t)
    return 'Te' if t.startswith('Te') else t

def _make_static_arrow_plot(atom_ids, atom_types, atom_positions, dz_full, out_path):
    z0 = atom_positions[:, 2].copy()

    # sort for clean vertical stack
    s = np.argsort(z0)
    atom_ids_s = atom_ids[s]
    atom_types_s = atom_types[s]
    z0_s = z0[s]
    dz_s = dz_full[s].copy()

    # normalize arrows
    max_abs = np.max(np.abs(dz_s))
    if max_abs > 0:
        dz_s = dz_s / max_abs

    style_types = np.array([_style_key(t) for t in atom_types_s], dtype=object)
    stationary_mask = np.isclose(dz_s, 0.0, atol=1e-10)
    moving_mask = ~stationary_mask

    # keep nonzero displacements visible while preserving sign
    dz_vis = dz_s.copy()
    dz_vis[moving_mask] = np.sign(dz_vis[moving_mask]) * np.maximum(np.abs(dz_vis[moving_mask]), MIN_VISIBLE_NONZERO)

    # keep atoms aligned on z as given by positions
    x0_s = np.zeros_like(z0_s, dtype=float)

    marker_sizes = {k: ATOM_SIZE_SCALE * (r ** 2) for k, r in ATOM_DISPLAY_RADII.items()}
    marker_colors = {'Mn': 'red', 'Bi': 'green', 'Te': 'blue'}

    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    ax.set_axis_off()

    # atoms: fade stationary sites so moving atoms stand out
    for t in ['Mn', 'Bi', 'Te']:
        m_stationary = (style_types == t) & stationary_mask
        m_moving = (style_types == t) & moving_mask
        if np.any(m_stationary):
            ax.scatter(
                x0_s[m_stationary], z0_s[m_stationary],
                s=marker_sizes[t], c=marker_colors[t], alpha=STATIONARY_ALPHA,
                linewidths=0, zorder=3
            )
        if np.any(m_moving):
            ax.scatter(
                x0_s[m_moving], z0_s[m_moving],
                s=marker_sizes[t], c=marker_colors[t], alpha=1.0,
                linewidths=0, zorder=3
            )

    # arrows (vertical only)
    if np.any(moving_mask):
        u = np.zeros_like(dz_s[moving_mask])
        v = PEAK_DISPLACEMENT * dz_vis[moving_mask]
        ax.quiver(
            x0_s[moving_mask], z0_s[moving_mask], u, v,
            angles='xy', scale_units='xy', scale=1,
            color=ARROW_COLOR, width=0.012, headwidth=3.8, headlength=2, headaxislength=2,
            zorder=4
        )

    # # light guide lines
    # for xx, zz, dzz in zip(x0_s[moving_mask], z0_s[moving_mask], dz_vis[moving_mask]):
    #     ax.plot([xx, xx], [zz, zz + PEAK_DISPLACEMENT * dzz], color='0.75', lw=2, zorder=2)

    pad_y = 4.0
    ax.set_xlim(-0.55, 0.55)
    ax.set_ylim(z0_s.min() - pad_y, z0_s.max() + pad_y)
    # ax.set_title(fontsize=10)

    legend_handles = [
        ax.scatter([], [], s=marker_sizes['Mn'], c=marker_colors['Mn'], linewidths=0),
        ax.scatter([], [], s=marker_sizes['Bi'], c=marker_colors['Bi'], linewidths=0),
        ax.scatter([], [], s=marker_sizes['Te'], c=marker_colors['Te'], linewidths=0),
    ]
    ax.legend(legend_handles, ['Mn', 'Bi', 'Te'], loc='upper right', frameon=False, fontsize=8)

    fig.savefig(out_path, bbox_inches='tight', pad_inches=0.02)
    plt.close(fig)

# ---- parse input ----
smodes_text_all = SMODES_FILE.read_text()
smodes_text = extract_irrep_section(smodes_text_all, IRREP_ANCHOR)
atom_ids, atom_types, atom_positions, mode_blocks = parse_atoms_and_modes(smodes_text)
dz_sym = build_symmetry_mode_displacements(atom_ids, mode_blocks)

print(f'Selected section: {IRREP_ANCHOR}')
print(f'Parsed {len(atom_ids)} atoms and {len(mode_blocks)} symmetry modes.')
print(f'Output dir: {OUT_DIR.resolve()}')

# ---- save one static image per symmetry mode ----
for mode_index, block in enumerate(mode_blocks, start=1):
    dz = dz_sym[mode_index - 1]
    species = sorted(set(str(t) for _, t, _ in block))
    species_tag = '_'.join(species)
    out = OUT_DIR / f'smodes_mode_{mode_index:02d}_{species_tag}.png'
    _make_static_arrow_plot(
        atom_ids, atom_types, atom_positions, dz,
        out_path=out
    )
    print('Saved:', out)

Selected section: Irrep T2-
Parsed 14 atoms and 4 symmetry modes.
Output dir: /Users/treycole/Repos/MBT-axion-phonon/qe_calcs/MnBi2Te4/phonon/pics
Saved: pics/smodes_mode_01_Mn.png
Saved: pics/smodes_mode_02_Bi.png
Saved: pics/smodes_mode_03_Te1.png
Saved: pics/smodes_mode_04_Te2.png


In [2]:
# Eigenmode coefficients in the symmetry-mode basis (same as phonon_gif.ipynb)
EIGVECS = np.array([
    [-0.36369331, -0.43130046, -0.88762667,  0.69114685],
    [-0.54129087,  0.50409351,  0.12255009,  0.24709458],
    [-0.75810732, -0.50196940,  0.04129069, -0.43020842],
    [-0.00215743,  0.55488410, -0.44203557, -0.52552927],
], dtype=float)

FREQ_THz = np.array([8.118075, 1.043247, 6.305090, 4.894563], dtype=float)

n_sym, n_atoms = dz_sym.shape
assert EIGVECS.shape[0] == n_sym, 'EIGVECS first axis must match number of symmetry modes'

for lam in range(EIGVECS.shape[1]):
    coeffs = EIGVECS[:, lam]
    dz_eigen = np.tensordot(coeffs, dz_sym, axes=(0, 0))

    out = OUT_DIR / f'phonon_eigenmode_{lam+1:02d}_f_{FREQ_THz[lam]:.3f}THz.png'
    _make_static_arrow_plot(
        atom_ids, atom_types, atom_positions, dz_eigen,
        title=f'Eigenmode {lam+1} | f = {FREQ_THz[lam]:.3f} THz',
        out_path=out
    )
    print('Saved:', out)

TypeError: _make_static_arrow_plot() got an unexpected keyword argument 'title'